#### Validando se o arquivo novo perde ou não dados e se repete as linhas:

In [ ]:
# === Importando Bibliotecas e consumindo a Base de Dados original ===
from copy import copy
from openpyxl.styles import PatternFill, Font
from openpyxl import load_workbook
import pandas as pd

# === NOVO (necessário p/ alinhar colunas fora de ordem e criar chaves robustas) ===
import re
import unicodedata


# === CONFIGURAÇÃO ===
arquivo_excel = r".\xlsx\Ativos e Desligados - 09-30-2025.xlsx"  # Primeiro arquivo...
arquivo_comparacao = r".\xlsx\Ativos e Desligados - 06-31-2025.xlsx"  # Segundo arquivo...

# === Arquivo(s) adicional(is) para inserir linhas novas no final ===
# O arquivo 12-31-2025 agora traz uma aba nova de desligados.
# O arquivo 04-01-2026 traz novo layout e a aba Sheet1 também é de desligados.
arquivos_adicionais = [
    r".\xlsx\Ativos e Desligados - 12-31-2025.xlsx",
    r".\xlsx\Ativos e Desligados - 04-01-2026.xlsx",
]

saida = r"..\..\..\..\Base de Dados Original\xlsx\Ativos e Desligados\Ativos e Desligados.xlsx"  # Arquivo final...


# === ABRIR O ARQUIVO ORIGINAL (para preservar layout) ===
wb = load_workbook(arquivo_excel)

# === LEITURA DAS PLANILHAS (pandas) ===
# Mantém compatibilidade com arquivos antigos de 4 abas e com arquivos novos de 2 abas.
pagina1 = 0
pagina2 = 1
pagina3 = 2
pagina4 = 3

abas_excel = pd.ExcelFile(arquivo_excel).sheet_names

def ler_planilha_pandas(caminho, indice):
    if indice >= len(abas_excel):
        return pd.DataFrame()
    return pd.read_excel(caminho, sheet_name=indice)

df1 = ler_planilha_pandas(arquivo_excel, pagina1)
df2 = ler_planilha_pandas(arquivo_excel, pagina2)
df3 = ler_planilha_pandas(arquivo_excel, pagina3)
df4 = ler_planilha_pandas(arquivo_excel, pagina4)


In [ ]:
# === FUNÇÃO DE LIMPEZA ===
def limpar(df):
    df = df.copy()
    df.columns = pd.Index(df.columns).map(lambda c: str(c).strip().lower())
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip().str.lower()
    return df

# === APLICAR LIMPEZA ===
df1, df2, df3, df4 = map(limpar, [df1, df2, df3, df4])

# === FUNÇÃO PARA PREPARAR DATAS ===
def preparar_datas(df, nome_coluna):
    if nome_coluna.lower() in df.columns:
        df[nome_coluna.lower()] = df[nome_coluna.lower()].astype(str)
        df[nome_coluna.lower()] = df[nome_coluna.lower()].str.replace('.', '/', regex=False)
        df[nome_coluna.lower()] = pd.to_datetime(df[nome_coluna.lower()], errors='coerce', dayfirst=True)
        df['ano'] = df[nome_coluna.lower()].dt.year
        df['mes'] = df[nome_coluna.lower()].dt.month
    return df

# === PREPARAR DATAS NAS PÁGINAS ===
df1 = preparar_datas(df1, 'Data_Admissão')
df3 = preparar_datas(df3, 'Data_Admissão')
df2 = preparar_datas(df2, 'Data_Dem')
df4 = preparar_datas(df4, 'Data_Dem')

# === FUNÇÃO DE COMPARAÇÃO (mantida) ===
def comparar_paginas(df_origem, df_destino, nome_origem, nome_destino, coluna_data):
    periodos_comuns = set(zip(df_origem['ano'], df_origem['mes'])) & set(zip(df_destino['ano'], df_destino['mes']))
    df_origem_periodo = df_origem[df_origem[['ano', 'mes']].apply(tuple, axis=1).isin(periodos_comuns)]

    faltando = df_origem_periodo.merge(df_destino, how='left', indicator=True)
    faltando = faltando[faltando['_merge'] == 'left_only'].drop(columns=['_merge'])
    faltando['pagina'] = nome_origem

    print(f"\n🔍 {nome_origem} → {nome_destino}")
    if faltando.empty:
        print("✅ Nenhuma linha diferente encontrada no mesmo período.")
    else:
        print(f"⚠️ {len(faltando)} linha(s) novas em {nome_origem} dentro do mesmo período de {nome_destino}.")

    return faltando


# === ABRIR O ARQUIVO ORIGINAL (mantém formatação e maiúsculas) ===
wb = load_workbook(arquivo_excel)
print("Abas encontradas:", wb.sheetnames)

# === Helpers de leitura/escrita preservando layout ===
def limpar_dados_aba(ws):
    if ws.max_row > 1:
        ws.delete_rows(2, ws.max_row - 1)

def atualizar_cabecalhos(ws, colunas):
    max_col_original = max(1, ws.max_column)
    modelo = ws.cell(row=1, column=min(max_col_original, max(1, len(colunas))))

    for c_idx, col in enumerate(colunas, start=1):
        celula = ws.cell(row=1, column=c_idx)
        if c_idx > max_col_original and modelo is not None:
            if modelo.has_style:
                celula._style = copy(modelo._style)
            celula.font = copy(modelo.font)
            celula.fill = copy(modelo.fill)
            celula.border = copy(modelo.border)
            celula.alignment = copy(modelo.alignment)
            celula.number_format = modelo.number_format
            celula.protection = copy(modelo.protection)
        celula.value = col

    for c_idx in range(len(colunas) + 1, ws.max_column + 1):
        ws.cell(row=1, column=c_idx).value = None

def escrever_dataframe_na_aba(ws, df):
    colunas = list(df.columns)
    atualizar_cabecalhos(ws, colunas)
    limpar_dados_aba(ws)

    for r_idx, row in enumerate(df.itertuples(index=False), start=2):
        for c_idx, valor in enumerate(row, start=1):
            ws.cell(row=r_idx, column=c_idx, value=None if pd.isna(valor) else valor)

# === FUNÇÃO PARA LER UMA ABA COMO TEXTO PURO (sem pandas lower) ===
def ler_aba_como_dataframe(workbook, nome_aba):
    ws = workbook[nome_aba]
    dados = []

    for row in ws.iter_rows(values_only=True):
        dados.append(list(row))

    if not dados:
        return pd.DataFrame()

    colunas = [str(c) if c is not None else f"Col_{i}" for i, c in enumerate(dados[0])]
    df = pd.DataFrame(dados[1:], columns=colunas)
    return df

def ler_aba_por_indice(workbook, indice):
    if indice >= len(workbook.sheetnames):
        return pd.DataFrame()
    return ler_aba_como_dataframe(workbook, workbook.sheetnames[indice])

# === Normalização de nomes e chaves ===
def _norm_txt(s: str) -> str:
    s = "" if s is None else str(s)
    s = s.strip().lower()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r"[^a-z0-9]+", "", s)
    return s

def _concept(col_name: str) -> str:
    texto = "" if col_name is None else str(col_name)
    n = _norm_txt(texto)

    # A coluna "   Cargo" no layout 04-01-2026 é código do cargo; "Cargo" é descrição.
    if texto != texto.lstrip() and n == "cargo":
        return "codcargo"

    # conceitos equivalentes (pra casar Data_Adm, Data Admissão, Entrada/Data etc.)
    if n in {"datadem", "datademissao", "datadesligamento", "saida"}:
        return "datademissao"
    if n in {"dataadm", "dataadmissao", "dataadmiss", "entrada", "data"}:
        return "dataadmissao"

    if n in {"npess", "nopess", "numpess", "numeropess", "npessoal"}:
        return "npess"
    if n in {"nomecompleto", "nome"}:
        return "nomecompleto"
    if n == "cpf":
        return "cpf"

    if n in {"sgemp", "subgrupoempregado", "subgrupoempregados", "subgrupoemp"}:
        return "subgrupoempregado"
    if n in {"uoabrev", "uoabreviacao"}:
        return "uoabrev"
    if n in {"denomuo", "uodescricao", "descricaouo", "denominacaouo"}:
        return "denomuo"

    return n

def _build_concept_map(df: pd.DataFrame) -> dict:
    m = {}
    for c in df.columns:
        cc = _concept(c)
        if cc not in m:
            m[cc] = c
    return m

def _clean_digits(v) -> str:
    if pd.isna(v):
        return ""
    s = str(v).strip()
    s = re.sub(r"\.0$", "", s)
    return re.sub(r"\D+", "", s)

def _clean_name(v) -> str:
    if pd.isna(v):
        return ""
    s = " ".join(str(v).strip().split())
    return s.lower()

def _make_key(df: pd.DataFrame) -> pd.Series:
    cmap = _build_concept_map(df)
    c_npess = cmap.get("npess")
    c_nome = cmap.get("nomecompleto")
    c_cpf = cmap.get("cpf")

    # O layout 04-01-2026 não traz CPF, então a chave principal precisa bater por pessoa+nome.
    if c_npess and c_nome:
        chave = df[c_npess].apply(_clean_digits) + "|" + df[c_nome].apply(_clean_name)
    elif c_npess and c_cpf:
        chave = df[c_npess].apply(_clean_digits) + "|" + df[c_cpf].apply(_clean_digits)
    elif c_npess:
        chave = df[c_npess].apply(_clean_digits)
    elif c_cpf and c_nome:
        chave = df[c_cpf].apply(_clean_digits) + "|" + df[c_nome].apply(_clean_name)
    elif c_cpf:
        chave = df[c_cpf].apply(_clean_digits)
    else:
        chave = df.astype(str).agg("|".join, axis=1)

    return chave.fillna("").astype(str)

def _coalesce_series(atual: pd.Series, nova: pd.Series, preferir_nova=False) -> pd.Series:
    atual_vazio = atual.isna() | atual.astype(str).str.strip().str.lower().isin(["", "nan", "none"])
    nova_vazia = nova.isna() | nova.astype(str).str.strip().str.lower().isin(["", "nan", "none"])

    if preferir_nova:
        return nova.where(~nova_vazia, atual)
    return atual.where(~atual_vazio, nova)

def _preparar_extra_para_destino(df_extra: pd.DataFrame, df_dest: pd.DataFrame) -> pd.DataFrame:
    cmap_dest = _build_concept_map(df_dest)
    resultado = pd.DataFrame(index=df_extra.index)

    for col in df_extra.columns:
        conceito = _concept(col)
        destino = cmap_dest.get(conceito, col)
        serie = df_extra[col]

        if destino in resultado.columns:
            # Se houver duas colunas candidatas ao mesmo destino, a coluna com nome exato vence.
            preferir_nova = _norm_txt(col) == _norm_txt(destino)
            resultado[destino] = _coalesce_series(resultado[destino], serie, preferir_nova=preferir_nova)
        else:
            resultado[destino] = serie

    return resultado

# === LER AS ABAS COM openpyxl (preserva conteúdo original) ===
df1 = ler_aba_por_indice(wb, 0)
df2 = ler_aba_por_indice(wb, 1)
df3 = ler_aba_por_indice(wb, 2)
df4 = ler_aba_por_indice(wb, 3)


# === FUNÇÃO PARA UNIR E ATUALIZAR UMA ABA PRESERVANDO LAYOUT E MAIÚSCULAS ===
def unir_e_atualizar(indice_destino, df_a, df_b, nome_destino):
    nome_aba = wb.sheetnames[indice_destino]
    ws = wb[nome_aba]

    colunas_destino = list(df_a.columns)
    for col in df_b.columns:
        if col not in colunas_destino:
            colunas_destino.append(col)

    df_a = df_a.reindex(columns=colunas_destino)
    df_b = df_b.reindex(columns=colunas_destino)

    df_a["origem"] = "1_ou_2"
    df_b["origem"] = "3_ou_4"

    df_unido = pd.concat([df_a, df_b], ignore_index=True)

    chaves = _make_key(df_unido)
    if chaves.astype(bool).any():
        df_unido["_key"] = chaves
        df_unido["prioridade"] = df_unido["origem"].apply(lambda x: 1 if x == "3_ou_4" else 0)
        df_unido = df_unido.sort_values(by="prioridade", ascending=True)

        antes = len(df_unido)
        df_unido = df_unido.drop_duplicates(subset=["_key"], keep="last")
        depois = len(df_unido)

        print(f"🧹 {antes - depois} duplicada(s) removida(s) em {nome_destino} — mantendo versão das páginas 3/4.")
        df_unido = df_unido.drop(columns=["origem", "prioridade", "_key"])
    else:
        df_unido = df_unido.drop(columns=["origem"])
        df_unido = df_unido.drop_duplicates()
        print(f"⚠️ Não foi possível montar chave robusta em {nome_destino}; usada deduplicação de linha inteira.")

    print(f"📄 {nome_destino}: {len(df_unido)} linhas únicas (mantendo dados das páginas 3/4 quando houver conflito).")

    escrever_dataframe_na_aba(ws, df_unido)


# === NOVA FUNÇÃO: REMOVER PESSOAS DA ABA 1 QUE TAMBÉM ESTÃO NA ABA 2 ===
def remover_duplicados_entre_abas(indice_aba1, indice_aba2):
    nome_aba1 = wb.sheetnames[indice_aba1]
    nome_aba2 = wb.sheetnames[indice_aba2]

    print(f"\n🔁 Comparando '{nome_aba1}' x '{nome_aba2}' para remover pessoas duplicadas da primeira aba...")

    ws1 = wb[nome_aba1]

    df1_atual = ler_aba_como_dataframe(wb, nome_aba1)
    df2_atual = ler_aba_como_dataframe(wb, nome_aba2)

    df1_atual["_key"] = _make_key(df1_atual)
    df2_atual["_key"] = _make_key(df2_atual)

    chaves_segunda = set(k for k in df2_atual["_key"].dropna().astype(str) if k)

    antes = len(df1_atual)
    df1_filtrado = df1_atual[~df1_atual["_key"].isin(chaves_segunda)].drop(columns=["_key"])
    removidas = antes - len(df1_filtrado)

    print(f"🧹 {removidas} linha(s) removida(s) da aba '{nome_aba1}' porque a mesma pessoa está na aba '{nome_aba2}' (mantendo só na segunda).")

    escrever_dataframe_na_aba(ws1, df1_filtrado)


# ==========================================================
# === INSERIR LINHAS NOVAS DE ARQUIVOS ADICIONAIS ==========
# ==========================================================

def _escolher_destino(df_extra: pd.DataFrame, wb_principal, nome_aba: str = "") -> int:
    nome_norm = _norm_txt(nome_aba)
    extra_concepts = set(_concept(c) for c in df_extra.columns)

    if "deslig" in nome_norm:
        return 1
    if "ativo" in nome_norm:
        return 0

    # No arquivo 04-01-2026 a aba de desligados veio como Sheet1.
    sinais_desligados = {"meds", "mtvmd", "denominacaodomotivodemedid"}
    if sinais_desligados & extra_concepts and "grpempregados" not in extra_concepts:
        return 1

    idxs = list(range(min(2, len(wb_principal.sheetnames))))
    melhor_idx = idxs[0]
    melhor_score = -1

    for idx in idxs:
        nome_dest = wb_principal.sheetnames[idx]
        df_dest = ler_aba_como_dataframe(wb_principal, nome_dest)
        dest_concepts = set(_concept(c) for c in df_dest.columns)

        inter = len(extra_concepts & dest_concepts)
        score = inter / max(1, len(extra_concepts))

        if score > melhor_score:
            melhor_score = score
            melhor_idx = idx

    return melhor_idx

def inserir_linhas_novas_no_wb(df_extra: pd.DataFrame, idx_destino: int):
    nome_dest = wb.sheetnames[idx_destino]
    ws_dest = wb[nome_dest]

    df_dest = ler_aba_como_dataframe(wb, nome_dest)
    if df_dest.empty:
        df_dest = pd.DataFrame(columns=list(df_extra.columns))

    df_extra = _preparar_extra_para_destino(df_extra, df_dest)

    # Garante colunas (união, mantendo ordem do destino e adicionando novas no fim)
    colunas = list(df_dest.columns)
    for c in df_extra.columns:
        if c not in colunas:
            colunas.append(c)

    df_dest = df_dest.reindex(columns=colunas)
    df_extra = df_extra.reindex(columns=colunas)

    # Cria chave e insere somente o que não existe ainda
    df_dest["_key"] = _make_key(df_dest)
    df_extra["_key"] = _make_key(df_extra)

    keys_dest = set(k for k in df_dest["_key"].dropna().astype(str) if k)
    novos = df_extra[df_extra["_key"].astype(str).ne("") & ~df_extra["_key"].isin(keys_dest)].copy()

    duplicadas_no_extra = novos.duplicated(subset=["_key"]).sum()
    if duplicadas_no_extra:
        novos = novos.drop_duplicates(subset=["_key"], keep="last")

    print(f"\n➕ Inserindo arquivo adicional em '{nome_dest}': {len(novos)} linha(s) nova(s).")
    if duplicadas_no_extra:
        print(f"🧹 {duplicadas_no_extra} duplicada(s) internas removida(s) antes da inserção.")

    df_final = pd.concat([df_dest, novos], ignore_index=True)
    df_final = df_final.drop(columns=["_key"])

    escrever_dataframe_na_aba(ws_dest, df_final)

def integrar_arquivo_adicional(caminho_arquivo: str):
    wb_extra = load_workbook(caminho_arquivo)
    print(f"\n📥 Lendo arquivo adicional: {caminho_arquivo}")
    print("   Abas no adicional:", wb_extra.sheetnames)

    for aba in wb_extra.sheetnames:
        df_extra = ler_aba_como_dataframe(wb_extra, aba)
        if df_extra.empty:
            continue

        idx_destino = _escolher_destino(df_extra, wb, aba)
        inserir_linhas_novas_no_wb(df_extra, idx_destino)


# === UNIR Página 1 + Página 3 ===
unir_e_atualizar(0, df1, df3, "Página 1 + Página 3")

# === UNIR Página 2 + Página 4 ===
unir_e_atualizar(1, df2, df4, "Página 2 + Página 4")

# === Integrar bases adicionais antes de remover duplicados entre abas ===
for arq in arquivos_adicionais:
    integrar_arquivo_adicional(arq)

# === AGORA COMPARA ABA 1 x ABA 2 E REMOVE DA 1 QUEM JÁ ESTÁ NA 2 ===
remover_duplicados_entre_abas(0, 1)


In [ ]:
# === RENOMEAR A SEGUNDA ABA PARA 'Desligados' ===
wb.sheetnames  # garante que a lista está atualizada
segunda_aba = wb.worksheets[1]
segunda_aba.title = "Desligados"
print("📝 Aba renomeada: Página 2 + Página 4 → Desligados")

# === REMOVER ABAS 3 e 4 (deixa só as duas primeiras) ===
while len(wb.sheetnames) > 2:
    nome = wb.sheetnames[-1]
    del wb[nome]
    print(f"🗑️ Aba removida: {nome}")

# === SALVAR O NOVO ARQUIVO ===
wb.save(saida)
print(f"\n✅ Arquivo final salvo preservando maiúsculas, conteúdo e visual em: {saida}")